## Агентский театр. «Мышеловка»
По трагедии Уильяма Шекспира «Гамлет», акт III, сцена 2.

Выполнил: Крижановский Данил.

Для начала, загрузим ключи из файла .env:

In [1]:
from dotenv import load_dotenv
import os

if load_dotenv(override=True):
    print(".env loaded")

folder_id = os.environ['folder_id']
api_key = os.environ['api_key']

.env loaded


In [2]:
from openai import OpenAI

# Создаем клиент OpenAI с настройками для Yandex Cloud
client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

### Клавдий

Короля играет [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). Веса загружаются в кэш, реплики генерируются локально.

In [3]:
import os
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
cache_dir = Path(os.environ.get("HF_HOME", Path(sys.prefix) / "hf-cache")) / "hub"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, cache_dir=cache_dir)
claudius_model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, cache_dir=cache_dir,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()
print(f"Клавдий: {model_id} · {device}")

D:\MAI\Сошников Промпт\labs\Lab1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1116 20:47:53.332000 1836 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:  10%|█         | 35/338 [00:00<00:01, 283.28it/s]

Loading weights:  19%|█▉        | 64/338 [00:00<00:01, 249.20it/s]

Loading weights:  26%|██▋       | 89/338 [00:00<00:01, 239.67it/s]

... (9 outputs left out)

Loading weights:  90%|████████▉ | 304/338 [00:01<00:00, 176.53it/s]

Loading weights:  96%|█████████▌| 323/338 [00:01<00:00, 171.45it/s]

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 198.79it/s]

Клавдий: Qwen/Qwen2.5-1.5B-Instruct · cuda


In [4]:
@torch.inference_mode()
def claudius_reply(public_scene):
    messages = [
        {"role": "system", "content": (
            "Ты Клавдий, король Дании из «Гамлета». При дворе говори сдержанно. "
            "Ответь одной короткой репликой по-русски, без подписи и ремарок."
        )},
        {"role": "user", "content": public_scene},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    output = claudius_model.generate(**inputs, max_new_tokens=64, do_sample=False)
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    line = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    if not line:
        raise ValueError("Клавдий не произнёс реплику")
    return line

### После представления. Горацио сверяется с текстом

Гамлет просит восстановить порядок событий. Горацио сам выбирает слова для поиска. Инструмент читает [локальную копию III акта, сцены 2](myshelovka.txt) и возвращает совпадение с соседними абзацами. Номера абзацев в выводе относятся к этой копии.

Уильям Шекспир, «Гамлет», перевод А. Кронеберга. [Источник текста](https://www.lib.ru/SHAKESPEARE/hamlet2.txt).

In [5]:
import json
import re
from pathlib import Path

scene_file = Path("myshelovka.txt")

def search_scene(query):
    paragraphs = re.split(r"\n\s*\n", scene_file.read_text(encoding="utf-8").strip())
    paragraphs = [" ".join(p.split()) for p in paragraphs]
    normalized = query.lower().replace("ё", "е")
    # В переводе Офелия использует настоящее время, а свидетель спрашивает о прошлом.
    if "корол" in normalized and re.search(r"вст[ао]|вскоч|подн[яи]|подъем", normalized):
        normalized = "король встает"
    words = re.findall(r"[а-яa-z]+", normalized)
    terms = {word[:5] for word in words if len(word) >= 3}
    matches = []
    for i, paragraph in enumerate(paragraphs):
        tokens = re.findall(r"[а-яa-z]+", paragraph.lower().replace("ё", "е"))
        overlap = terms.intersection(word[:5] for word in tokens)
        if overlap and (len(terms) == 1 or len(overlap) >= 2):
            phrase = " ".join(words) in " ".join(tokens)
            matches.append((len(overlap), phrase, i))
    if not matches:
        return "Совпадений нет. Попробуй другие слова из сцены."

    excerpts = []
    best = max(hit[:2] for hit in matches)
    for _, _, i in [hit for hit in matches if hit[:2] == best][:2]:
        first, last = max(0, i - 4), min(len(paragraphs), i + 4)
        excerpts.append(f"Абзацы {first + 1}–{last}:\n" + "\n".join(paragraphs[first:last]))
    return "\n\n".join(excerpts)

search_tool = {
    "type": "function",
    "name": "search_scene",
    "description": "Найти в русском тексте сцены реплику по фразе или ключевым словам. Возвращает и соседние реплики.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Короткая поисковая фраза по-русски"}},
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [6]:
def ask_horatio(question, search_hint=""):
    messages = [{"role": "user", "content": question}]
    model = f"gpt://{folder_id}/qwen3.6-35b-a3b"
    for attempt in range(3):
        request = client.responses.create(
            model=model, input=messages, tools=[search_tool], tool_choice="required",
            instructions="Выбери для поиска короткую фразу. При пустом результате измени запрос. " + search_hint,
        )
        call = next((item for item in request.output if item.type == "function_call"), None)
        if call is None or call.name != "search_scene":
            raise ValueError("Горацио не вызвал поиск")
        query = json.loads(call.arguments)["query"]
        excerpt = search_scene(query)
        print(f"Запрос: {query}\n{excerpt}\n")
        messages.extend([
            {"type": "function_call", "name": call.name, "arguments": call.arguments, "call_id": call.call_id},
            {"type": "function_call_output", "call_id": call.call_id, "output": excerpt},
        ])
        if excerpt.startswith("Абзацы "):
            reply = client.responses.create(
                model=model,
                instructions="Ты Горацио. Ответь на оба вопроса Гамлета по найденным строкам: назови имя свидетеля из текста и цель убийцы. Два коротких предложения.",
                input=f"Отрывок из пьесы:\n{excerpt}\nВопрос Гамлета: {question}",
            )
            answer = reply.output_text.strip()
            if not answer or "<tool_call>" in answer:
                raise ValueError("Горацио не выдал реплику по найденному отрывку")
            return answer
    return "Друг мой, я не нашёл этой строки и не берусь ответить по памяти."

### Макроплан спектакля

Режиссёр читает сцену Шекспира и намечает события, ремарки и задачи для трёх действий. Исполнителя каждого хода он выбирает позже, когда уже прозвучала предыдущая реплика.

In [7]:
from pydantic import BaseModel

class Ход(BaseModel):
    ремарка: str
    задача: str

class Действие(BaseModel):
    название: str
    событие: str
    ходы: list[Ход]

class Партитура(BaseModel):
    действия: list[Действие]

director = f"gpt://{folder_id}/deepseek-v4.1-flash"
plan_request = client.responses.create(
    model=director,
    instructions=(
        "Ты режиссёр камерной постановки «Мышеловки». Составь по источнику план трёх действий "
        "по-русски. Для каждого хода дай короткую видимую ремарку перед репликой и игровую задачу "
        "без назначения исполнителя. В плане не пиши готовых реплик. "
        "I: до прихода двора Гамлет просит наблюдать за королём, Горацио соглашается. "
        "II: Клавдий спрашивает название пьесы, Гамлет отвечает «Мышеловка»; "
        "следом Луциан вливает яд спящему в ухо, Гамлет поясняет цель убийцы. "
        "Отрази действия Луциана и то, что Офелия замечает подъём короля, через ремарки. "
        "Последним говорит Клавдий: требует света и уходит. "
        "III: двор ушёл, Гамлет спрашивает, кто заметил подъём короля и зачем отравили Гонзаго. "
        "Горацио сверяется с текстом. Никто не признаётся в преступлении. "
        "Названия действий без слова «Действие» и номера. В игровых задачах прямо укажи "
        "требование света и цель отравления Гонзаго — завладеть его царством. "
        "Во II действии ровно четыре отдельные речевые задачи: вопрос короля о названии; ответ принца с названием «Мышеловка»; комментарий принца к отравлению; требование короля дать свет. Не выделяй немые действия в отдельный ход. Для начала каждого действия укажи инициатора и адресата в ремарке в форме «Имя, обращаясь к …». Для ответов укажи только видимое действие. В I и III действиях по два хода."
    ),
    input=scene_file.read_text(encoding="utf-8"),
    text={"format": {"type": "json_schema", "name": "play_plan", "schema": Партитура.model_json_schema()}},
)
score = Партитура.model_validate_json(plan_request.output_text)
if [len(act.ходы) for act in score.действия] != [2, 4, 2]:
    raise ValueError("План должен содержать три действия с 2, 4 и 2 ходами")

for number, act in enumerate(score.действия, 1):
    print(f"{number}. {act.название} — {act.событие}")
    for beat in act.ходы:
        print(f"   {beat.задача}")

1. Перед спектаклем — До прихода двора Гамлет просит Горацио наблюдать за королём во время представления; Горацио соглашается.
   Попросить Горацио во время пьесы неотрывно наблюдать за королём и запомнить его поведение.
   Согласиться и дать понять, что будет следить за королём.
2. Мышеловка — При дворе играют пьесу; Клавдий спрашивает название, Луциан отравляет спящего, Офелия замечает подъём короля, а Клавдий требует света и уходит.
   Спросить у Гамлета название пьесы.
   Ответить королю названием «Мышеловка».
   Пояснить, что убийца отравил Гонзаго, чтобы завладеть его царством.
   Потребовать дать свет и покинуть зал.
3. После ухода двора — Двор ушёл; Гамлет расспрашивает Горацио о подъёме короля и причине отравления Гонзаго; Горацио сверяется с текстом; никто не признаётся в преступлении.
   Спросить, кто заметил подъём короля и зачем отравили Гонзаго — чтобы завладеть его царством.
   Свериться с текстом и дать понять, что никто не признаётся в преступлении.


### Спектакль

Явное обращение в ремарке передаёт ход тому, кто обращается. В остальных случаях режиссёр получает список присутствующих, сыгранные в этом действии реплики и ближайшую ремарку. Выбранный персонаж отвечает с учётом того, что сам слышал. После ухода двора Горацио может свериться с текстом пьесы.

In [8]:
import io
from contextlib import redirect_stdout

actor_models = {"Гамлет": director, "Горацио": f"gpt://{folder_id}/qwen3.6-35b-a3b"}
voices = {
    "Гамлет": "Наедине с другом говори прямо, при дворе отвечай с иронией.",
    "Горацио": "Говори спокойно, различай наблюдение и догадку.",
}

def choose_speaker(audience, dialogue, beat, scene_goal):
    # В явной ремарке «Гамлет, обращаясь к Горацио, …» говорит Гамлет.
    marked = re.findall(r"\b(Гамлет|Горацио|Клавдий),?\s+обращаясь\b", beat.ремарка)
    for actor in reversed(marked):
        if actor in audience:
            return actor
    context = "\n".join([f"Сюжет действия: {scene_goal}", *dialogue, beat.ремарка, f"Задача хода: {beat.задача}"])
    selection = client.responses.create(
        model=director, input=context,
        instructions=(
            "Ты театральный режиссёр. Учитывай, к кому обратились в последней реплике, "
            "что происходит на сцене и кто может выполнить задачу этого хода. "
            "Клавдий — король и зритель, Гамлет — принц и устроитель пьесы, Горацио — свидетель. Выбери говорящего: " + ", ".join(audience)
        ),
        text={"format": {"type": "json_schema", "name": "casting", "schema": {
            "type": "object", "properties": {"speaker": {"type": "string", "enum": audience}},
            "required": ["speaker"], "additionalProperties": False,
        }}},
    )
    name = json.loads(selection.output_text)["speaker"]
    if name not in audience:
        raise ValueError(f"Персонаж вне сцены: {name}")
    return name


def perform(score):
    memory = {name: [] for name in ["Клавдий", *actor_models]}
    script, search_log = [], io.StringIO()
    for number, act in enumerate(score.действия, 1):
        audience = ["Гамлет", "Горацио", "Клавдий"] if number == 2 else ["Гамлет", "Горацио"]
        script.append(f"Действие {number}. {act.название}")
        dialogue = []
        previous_line = ""
        for beat in act.ходы:
            name = choose_speaker(audience, dialogue, beat, act.событие)
            cue = f"({beat.ремарка})"
            context = "\n".join([*memory[name], cue, f"Твоя задача: {beat.задача}"])
            if name == "Клавдий":
                if beat is act.ходы[0]:
                    context += '\nТвои слова к Гамлету: А как называется пьеса?'
                elif beat is act.ходы[-1]:
                    context += '\nОпорная реплика роли: «Посветите мне! Идем!» Потребуй света и вели двору уходить.'
                line = claudius_reply(context)
            elif name == "Горацио" and number == 3 and previous_line:
                with redirect_stdout(search_log):
                    line = ask_horatio(previous_line, 'Сверь реакцию двора по словам перевода «король встает».')
            else:
                reply = client.responses.create(
                    model=actor_models[name], input=context,
                    instructions=f"Ты {name}. {voices[name]} Дай одну короткую реплику по-русски без подписи.",
                )
                line = reply.output_text.strip()
            if not line:
                raise ValueError(f"Действие {number}: нет реплики персонажа {name}")
            spoken = f"{name}: {line}"
            dialogue.extend([cue, spoken])
            script.extend([cue, spoken])
            for listener in audience:
                memory[listener].extend([cue, spoken])
            previous_line = line
        script.append("")
    return "\n".join(script).strip(), search_log.getvalue()

performance, horatio_log = perform(score)
print(performance)

Действие 1. Перед спектаклем
(Гамлет, обращаясь к Горацио, отводит его в сторону и указывает глазами на место, где должен сидеть король.)
Гамлет: Горацио, во время пьесы глаз с короля не спускай и запомни каждую его гримасу.
(Горацио, обращаясь к Гамлету, коротко кивает и переводит взгляд на королевское место.)
Горацио: Буду пристально смотреть. Отмечу лишь увиденное, не подменяя факты догадками.

Действие 2. Мышеловка
(Клавдий, обращаясь к Гамлету, подаётся вперёд в кресле и указывает на сцену.)
Клавдий: А как называется пьеса?
(Гамлет, обращаясь к Клавдию, поворачивается к сцене и делает короткий жест в сторону актёров.)
Гамлет: «Мышеловка», ваше величество: в ней ловят не только мышей.
(Луциан, обращаясь к спящему на сцене Гонзаго, наклоняется к его уху и вливает яд; Гамлет, обращаясь к собравшимся, указывает на сцену.)
Гамлет: Убийца влил яд в ухо спящему Гонзаго — не из любви к искусству, а чтобы завладеть его царством.
(Офелия, обращаясь к Гамлету, привстаёт и указывает на короля

In [9]:
print(horatio_log)

Запрос: король встает
Абзацы 174–181:
Он вливает яд в ухо спящего.
Гамлет
Он отравляет его в саду, чтобы завладеть его царством. Имя его - Гонзаго. История налицо: она превосходно описана по-итальянски. Вы сейчас увидите, как убийца вкрадется в любовь супруги Гонзаго.
Офелия
Король встает.
Гамлет
Как? Испуган ложною тревогой?
Королева


